# Temporal Interpolation of Weather-Radar Volumes by Advection Morphing

**Reconstructing an intermediate radar volume between two scans, on native gate geometry**

*Scott Collis · C-SAPR / MC3E · 20 May 2011 squall line*

---

A scanning weather radar delivers a full 3-D volume only every few minutes — C-SAPR at
the ARM Southern Great Plains site took roughly one volume every 7 minutes during MC3E.
For animation, data assimilation, or matching a radar to a faster-sampling instrument we
often want a volume *between* two real scans. Simply averaging the two bracketing volumes
smears fast-moving convection into a ghost that is in neither the right place at either
time nor anywhere sensible in between.

This notebook builds an intermediate volume **t₂** from two observed volumes **t₁** and
**t₃** by *advection morphing*: estimate how the echo is moving, push t₁ forward and t₃
backward along that motion to the target time, and blend. The motion is estimated with
**dense optical flow**, and — crucially — the final interpolation is done **on the radar's
own (azimuth, range, elevation) geometry**, one gate at a time, rather than on an
intermediate Cartesian grid. That keeps the output a true radar volume with the same
sweeps and gate spacing as a real scan.

The method is validated by leave-one-out: we hold out the *real* middle volume, reconstruct
it from its neighbours, and score the reconstruction gate-by-gate against the truth.

### What's in here
1. **Data** — fetching three consecutive C-SAPR surveillance volumes from the ARM Live archive
2. **Motion estimation** — why a single advection vector fails for an MCS, and dense optical flow as the fix
3. **The morph** — semi-Lagrangian forward/backward warping and the time blend (including the sign convention that is easy to get wrong)
4. **Native-geometry interpolation** — doing the morph per gate in (az, range) space
5. **Validation** — leave-one-out reconstruction of the real t₂, scored against baselines
6. **Animation** — filling the gaps between real volumes with smooth intermediate frames

### Requirements
`pyart` (Py-ART ≥ 2.0), `scikit-image` (TV-L1 optical flow), `scipy`, `numpy`, `matplotlib`.


## 1. Data — three consecutive C-SAPR volumes

We use the ARM datastream **`sgpcsaprsurI7.00`** — the C-band Scanning ARM Precipitation
Radar in surveillance mode at the Nardin, OK site (I7), delivered as one MDV tar per volume
(~190 MB each). The filename timestamp `HHMMSS` is the volume **end** time.

For the 20 May 2011 squall line we take three volumes bracketing ~11:31 UTC. Each volume
has 17 sweeps (0.75°–42° elevation), 6120 rays × 983 gates, 120 m gate spacing out to
~118 km.

> **Access note.** `sgpcsaprsurI7.00` downloads from the ARM Live `saveData` endpoint with
> `user=<username>:<token>`. Get a free token at <https://adc.arm.gov/armlive/> (Login →
> account settings). The processed CMAC netCDF variants (`sgpcsaprsurcmacI7.c0`) are
> catalogued but order-only, so we use the raw MDV surveillance stream here.


In [ ]:
import os, glob, tarfile, datetime, getpass
import numpy as np
import pyart

# --- ARM Live credentials (never hard-code a token in a shared notebook) ---
ARM_USER  = os.environ.get("ARM_USERNAME") or "your_arm_username"
ARM_TOKEN = os.environ.get("ARM_TOKEN")    or None   # or getpass.getpass("ARM token: ")

DATASTREAM = "sgpcsaprsurI7.00"
# volume END timestamps (UTC) for the three bracketing scans of the 11:31 case
VOL_STAMPS = ["20110520.112734", "20110520.113434", "20110520.114134"]
DATADIR = "data"; os.makedirs(DATADIR, exist_ok=True)

In [ ]:
import requests

QUERY = "https://adc.arm.gov/armlive/data/query"
SAVE  = "https://adc.arm.gov/armlive/data/saveData"

def fetch_arm_file(fname, user, token, outdir=DATADIR):
    """Download one ARM Live file to outdir; returns local path (skips if present)."""
    dest = os.path.join(outdir, fname)
    if os.path.exists(dest):
        return dest
    url = f"{SAVE}/user/{user}:{token}/{fname}"
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    with open(dest, "wb") as f:
        f.write(r.content)
    return dest

def load_volume(stamp, user=ARM_USER, token=ARM_TOKEN):
    """Fetch + extract the MDV tar for one volume stamp, return a Py-ART Radar."""
    tar_name = f"{DATASTREAM}.{stamp}.mdv.tar"
    if token:                                  # download if we have a token
        fetch_arm_file(tar_name, user, token)
    tpath = os.path.join(DATADIR, tar_name)
    with tarfile.open(tpath) as tf:
        tf.extractall(DATADIR)                 # -> data/sur/YYYYMMDD/HHMMSS.mdv
    ymd = stamp.split(".")[0]; hms = stamp.split(".")[1]
    mdv = os.path.join(DATADIR, "sur", ymd, f"{hms}.mdv")
    return pyart.io.read_mdv(mdv)

In [ ]:
# Load the three volumes  (t1 = earliest, t2 = middle/held-out truth, t3 = latest)
radars = {name: load_volume(stamp) for name, stamp in zip(["t1","t2","t3"], VOL_STAMPS)}

def volume_midtime(radar):
    """Mean ray time of the volume -> a single representative UTC datetime."""
    base = datetime.datetime.strptime(radar.time["units"].split("since")[1].strip(),
                                      "%Y-%m-%dT%H:%M:%SZ")
    return base + datetime.timedelta(seconds=float(np.mean(radar.time["data"])))

midtimes = {k: volume_midtime(v) for k, v in radars.items()}
for k in ["t1","t2","t3"]:
    print(f"{k}: {midtimes[k]:%H:%M:%S} UTC   "
          f"{radars[k].nsweeps} sweeps, {radars[k].nrays}x{radars[k].ngates} gates")
dt12 = (midtimes["t2"]-midtimes["t1"]).total_seconds()
dt23 = (midtimes["t3"]-midtimes["t2"]).total_seconds()
print(f"\ngaps: t1->t2 {dt12:.0f} s, t2->t3 {dt23:.0f} s")

The three volumes, at 11:24:47, 11:31:47, and 11:38:47 UTC, span a leading convective
line with trailing stratiform precipitation — a classic MCS structure moving toward the
east-northeast.

## 2. Estimating the motion

### Why a single advection vector fails

The intuitive approach — cross-correlate the whole t₁ and t₃ reflectivity fields to find one
bulk shift — **fails badly for an MCS**. The broad stratiform region is large, bright, and
nearly stationary in *pattern*, so it dominates a whole-domain correlation and pins the
estimate at zero shift, even while the embedded convective cells are moving at ~20 m/s.

We confirmed this directly: whole-domain phase and normalized cross-correlation both peak at
zero lag, and only when the correlation is restricted to the convective subregion does a real
displacement (~20 m/s toward the northwest, from the storm-relative frame) emerge. A single
rigid vector cannot represent a scene where different features move differently.

The difference field t₃ − t₁ shows the problem and the opportunity: a clear dipole
(loss where echo *was*, gain where it *went*) traces the local motion, and it varies across
the domain.

![t1, t3, and their difference — the motion dipole](figures/grid_difference.png)

### Dense optical flow

The fix is a **spatially varying** motion field. We use **TV-L1 optical flow**
(`skimage.registration.optical_flow_tvl1`) on the reflectivity, which returns a motion vector
at *every* grid cell while a total-variation penalty keeps the field spatially coherent. This
is cleaner than tile-based TREC (tracking radar echoes by correlation), which is noisy in
low-texture regions.

Optical flow is computed on Cartesian CAPPIs (constant-altitude plan-position indicators)
built with `pyart.map.grid_from_radars`. Because storm motion changes with height, we grid a
stack of levels (1–8 km) and estimate flow **per level** — low levels move at ~11–13 m/s,
weakening to a few m/s aloft.

![Optical-flow motion field and speed](figures/optical_flow_field.png)


In [ ]:
from skimage.registration import optical_flow_tvl1

# --- grid the volumes to a Cartesian stack for flow estimation ---
GRID_SHAPE  = (8, 241, 241)                       # (z, y, x): 8 levels 1-8 km
GRID_LIMITS = ((1000., 8000.), (-120000., 120000.), (-120000., 120000.))

def grid_volume(radar):
    g = pyart.map.grid_from_radars(
        radar, grid_shape=GRID_SHAPE, grid_limits=GRID_LIMITS,
        fields=["reflectivity"], weighting_function="Barnes2",
        roi_func="dist_beam", min_radius=1000.)
    return g

g1, g2, g3 = (grid_volume(radars[k]) for k in ["t1","t2","t3"])
zlev_m = g1.z["data"]; y_m = g1.y["data"]; x_m = g1.x["data"]
Z1c = np.ma.filled(g1.fields["reflectivity"]["data"], np.nan)
Z2c = np.ma.filled(g2.fields["reflectivity"]["data"], np.nan)   # truth, for validation only
Z3c = np.ma.filled(g3.fields["reflectivity"]["data"], np.nan)

In [ ]:
FLOOR = -10.0
def norm_refl(A, lo=0., hi=60.):
    """Map reflectivity to [0,1] with non-echo pushed to the floor, for optical flow."""
    return (np.clip(np.where(np.isfinite(A), A, FLOOR), lo, hi) - lo) / (hi - lo)

def level_displacements(Za, Zb):
    """Per-level ECHO DISPLACEMENT (meters, a->b) as two (z,y,x) arrays (east, north).

    optical_flow_tvl1(moving, reference) returns (v, u) mapping *reference -> moving*,
    so the echo displacement from a to b is D = -flow.
    """
    dy = y_m[1]-y_m[0]; dx = x_m[1]-x_m[0]
    U = np.zeros_like(Za); V = np.zeros_like(Za)
    for kz in range(Za.shape[0]):
        fv, fu = optical_flow_tvl1(norm_refl(Za[kz]), norm_refl(Zb[kz]),
                                   attachment=15, tightness=0.3, num_warp=5, num_iter=10)
        V[kz] = (-fv) * dy      # north displacement (m) over the a->b interval
        U[kz] = (-fu) * dx      # east  displacement (m)
    return U, V

# pairwise flow: shorter gaps give better flow than spanning the full t1->t3 interval
U12, V12 = level_displacements(Z1c, Z2c)
U23, V23 = level_displacements(Z2c, Z3c)
spd = np.hypot(U12[1], V12[1]) / dt12          # 2 km level, m/s
echo = Z1c[1] > 5
print(f"2 km motion, t1->t2:  median {np.median(spd[echo]):.1f} m/s")

## 3. The morph — semi-Lagrangian warping and the time blend

With a displacement field **D** (echo motion from the earlier to the later volume) we build
the intermediate volume at fractional time **α ∈ (0,1)** by *warping both neighbours to the
target time and blending*:

- **Forward-warp** the earlier volume: sample it at `x + α·D` — where an echo *now at x*
  came *from*.
- **Backward-warp** the later volume: sample it at `x − (1−α)·D` — where an echo now at x
  is *going to*.
- **Blend** by the time weights: `(1−α)·earlier_warped + α·later_warped`.

This is a semi-Lagrangian scheme: for each output location we trace the trajectory back to
where the source data lives and interpolate there.

> ### ⚠️ The sign trap
> The single easiest thing to get wrong is the warp sign — flip it and the reconstruction
> is *worse* than a naïve average, because the echo is pushed the wrong way. The reliable
> check is **end-to-end**: warp t₁ by the *full* +D and see whether it reproduces t₃. It
> does (and −D does not). So the forward warp samples the source at **+α·D**. We verified
> this in both Cartesian and native geometry — get the sign from the validation, not from
> intuition.

For a first look we do the morph on the 2 km CAPPI, then validate against the held-out real
t₂. The morph beats a naïve linear average on every metric:

| method | RMSE (dBZ) | CC |
|---|---|---|
| naïve linear average | 6.64 | 0.805 |
| **optical-flow morph** | **5.55** | **0.859** |
| persistence (t₁) | 8.64 | 0.715 |

![Cartesian leave-one-out validation of t2](figures/cartesian_validation.png)


In [ ]:
from scipy.ndimage import map_coordinates

def morph_cartesian(A, B, U, V, alpha, dt, zlev=1):
    """Intermediate CAPPI level at fractional time alpha, from A (earlier) and B (later).
    U,V are TOTAL a->b displacements in meters; convert to pixels for map_coordinates."""
    dy = y_m[1]-y_m[0]; dx = x_m[1]-x_m[0]
    Dr = V[zlev]/dy; Dc = U[zlev]/dx                 # displacement in pixels
    ny, nx = A[zlev].shape
    row, col = np.mgrid[0:ny, 0:nx].astype(float)
    def warp(src, dr, dc):
        return map_coordinates(np.where(np.isfinite(src), src, FLOOR),
                               [row+dr, col+dc], order=1, mode="constant", cval=FLOOR)
    fwd = warp(A[zlev],  alpha*Dr,      alpha*Dc)     # sample earlier at +alpha*D
    bwd = warp(B[zlev], -(1-alpha)*Dr, -(1-alpha)*Dc) # sample later   at -(1-alpha)*D
    return (1-alpha)*fwd + alpha*bwd

# reconstruct t2 at alpha = dt12/(dt12+dt23) and score vs truth Z2c
alpha_t2 = dt12/(dt12+dt23)
recon = morph_cartesian(Z1c, Z3c, *level_displacements(Z1c, Z3c), alpha_t2, dt12+dt23, zlev=1)
truth = Z2c[1]; m = truth > 5
rmse = np.sqrt(np.nanmean((recon[m]-truth[m])**2))
lin  = 0.5*(np.where(np.isfinite(Z1c[1]),Z1c[1],FLOOR)+np.where(np.isfinite(Z3c[1]),Z3c[1],FLOOR))
rmse_lin = np.sqrt(np.nanmean((lin[m]-truth[m])**2))
print(f"morph RMSE {rmse:.2f} dBZ   vs linear-avg {rmse_lin:.2f} dBZ")

## 4. Native-geometry interpolation — one gate at a time

The Cartesian morph above is a useful validation, but its output is a *grid*, not a radar
volume. The project goal is a genuine intermediate **volume** — same sweeps, same
(azimuth, range) sampling as a real scan — so downstream tools see it as an ordinary
CfRadial file. We therefore do the interpolation **directly on the radar's native geometry**,
one gate at a time.

For every gate of the target volume, at its physical location (x, y, z):

1. **Look up the local motion** (u, v) from the height-resolved optical-flow field
   (a `RegularGridInterpolator` over the (z, y, x) flow stack).
2. **Advect** that point to where the echo sat at t₁ — `(x + α·D, y + α·D)` — and where it
   goes at t₃ — `(x − (1−α)·D, y − (1−α)·D)` — using the **same validated +D forward-warp
   sign**.
3. **Convert each advected point to native coordinates** for the *same sweep*: meteorological
   azimuth `atan2(x, y) mod 360°` and slant range `ground_range / cos(elevation)`.
4. **Sample** t₁ and t₃ at those (azimuth, range) positions (bilinear, with azimuth wrapped
   across 0°/360°), and **blend** by the time weights.

Advection is applied in the *horizontal*; each gate stays on its own sweep (constant
elevation), which is the right approximation for the near-horizontal low-level scans that
carry most of the precipitation echo.


In [ ]:
from scipy.interpolate import RegularGridInterpolator

def build_flow_interpolators(U, V, echo_mask):
    """RegularGridInterpolators for east/north displacement over (z,y,x).
    Non-echo cells are filled with the per-level mean so the field is defined everywhere."""
    Uf, Vf = U.copy(), V.copy()
    for kz in range(U.shape[0]):
        for F in (Uf, Vf):
            lvl, m = F[kz], echo_mask[kz]
            lvl[~m] = lvl[m].mean() if m.any() else 0.0
    kw = dict(bounds_error=False, fill_value=None)
    return (RegularGridInterpolator((zlev_m, y_m, x_m), Uf, **kw),
            RegularGridInterpolator((zlev_m, y_m, x_m), Vf, **kw))

def sweep_sampler(radar, k, field):
    """Return (az_sorted, rng, values, order) for sweep k, ready for bilinear sampling."""
    s0, s1 = radar.get_start_end(k)
    az = radar.azimuth["data"][s0:s1+1]
    rng = radar.range["data"]
    data = np.ma.filled(radar.fields[field]["data"][s0:s1+1].astype(float), np.nan)
    order = np.argsort(az)
    return az[order], rng, data[order], order

def sample_native(az_q, rng_q, sampler):
    """Bilinear sample a sweep at query azimuth (deg) / slant range (m), wrapping azimuth."""
    az_s, rng_s, vals, _ = sampler
    # tile azimuth by +-360 so queries near 0/360 interpolate across the seam
    az_ext = np.concatenate([az_s-360, az_s, az_s+360])
    val_ext = np.concatenate([vals, vals, vals], axis=0)
    rgi = RegularGridInterpolator((az_ext, rng_s), val_ext,
                                  bounds_error=False, fill_value=np.nan)
    return rgi(np.column_stack([az_q.ravel(), rng_q.ravel()])).reshape(az_q.shape)

In [ ]:
def interp_native_volume(rA, rB, fU, fV, alpha, field="reflectivity"):
    """Reconstruct a full volume at fractional time alpha between rA (earlier) and rB (later),
    on rA's native geometry. fU,fV give TOTAL a->b displacement (m) as f(z,y,x)."""
    out = np.full((rA.nrays, rA.ngates), np.nan)
    gx_all, gy_all, gz_all = rA.gate_x["data"], rA.gate_y["data"], rA.gate_z["data"]
    for k in range(rA.nsweeps):
        s0, s1 = rA.get_start_end(k)
        elev = np.deg2rad(rA.fixed_angle["data"][k]); ce = max(np.cos(elev), 1e-3)
        gx, gy, gz = gx_all[s0:s1+1], gy_all[s0:s1+1], gz_all[s0:s1+1]
        zc = np.clip(gz, zlev_m[0], zlev_m[-1])
        pts = np.column_stack([zc.ravel(), gy.ravel(), gx.ravel()])
        Du = fU(pts).reshape(gx.shape); Dv = fV(pts).reshape(gx.shape)
        # advect to t1 (+alpha D) and t3 (-(1-alpha) D) positions
        xa, ya = gx + alpha*Du,     gy + alpha*Dv
        xb, yb = gx - (1-alpha)*Du, gy - (1-alpha)*Dv
        to_azr = lambda xs, ys: (np.rad2deg(np.arctan2(xs, ys)) % 360, np.hypot(xs, ys)/ce)
        aza, ra = to_azr(xa, ya); azb, rb = to_azr(xb, yb)
        va = sample_native(aza, ra, sweep_sampler(rA, k, field))
        vb = sample_native(azb, rb, sweep_sampler(rB, k, field))
        both = np.isfinite(va) & np.isfinite(vb)
        out[s0:s1+1] = np.where(both, (1-alpha)*va + alpha*vb,
                                np.where(np.isfinite(va), va, vb))
    return out

# height-resolved flow interpolators for the t1<->t3 span (for leave-one-out t2)
echo_mask = (Z1c > 5) | (Z3c > 5)
U13, V13 = level_displacements(Z1c, Z3c)
fU13, fV13 = build_flow_interpolators(U13, V13, echo_mask)
R2_native = interp_native_volume(radars["t1"], radars["t3"], fU13, fV13, alpha_t2)
print("reconstructed native t2, finite fraction:", np.isfinite(R2_native).mean())

## 5. Validation — leave-one-out, scored gate-by-gate

We compare the native reconstruction against the **real** held-out t₂, gate for gate, over
all 17 sweeps (~3.8 M echo gates, truth > 5 dBZ). Baselines are computed *fairly* by
resampling t₁ and t₃ onto t₂'s exact geometry with **zero** advection, so every method is
scored on the identical set of gates.

| method | RMSE (dBZ) | MAE | bias | CC |
|---|---|---|---|---|
| persistence t₁→2 | 7.29 | 4.44 | −0.72 | 0.774 |
| persistence t₃→2 | 6.60 | 4.04 | −0.18 | 0.815 |
| linear avg (no advection) | 6.11 | 3.55 | −0.50 | 0.833 |
| **advection morph (native)** | **5.86** | **3.51** | **−0.28** | **0.844** |

The morph wins on every metric. The margin over linear averaging is modest (~4% RMSE) and
this is *physical*: near-range gates are small and the storm barely moves relative to gate
spacing there, so advection contributes little; its value concentrates at far range where
gates are large and the displacement spans several gate-widths. Residuals (bottom-right)
localize at cell edges — storm growth and decay, which a purely kinematic morph cannot
reproduce — not at bulk position.

![Native-geometry leave-one-out validation](figures/native_ppi_validation.png)


In [ ]:
def resample_to_geometry(rSrc, rTgt, field="reflectivity"):
    """Sample rSrc onto rTgt's native (az,range) geometry with NO advection (fair baseline)."""
    out = np.full((rTgt.nrays, rTgt.ngates), np.nan)
    gx_all, gy_all = rTgt.gate_x["data"], rTgt.gate_y["data"]
    for k in range(rTgt.nsweeps):
        s0, s1 = rTgt.get_start_end(k)
        elev = np.deg2rad(rTgt.fixed_angle["data"][k]); ce = max(np.cos(elev), 1e-3)
        gx, gy = gx_all[s0:s1+1], gy_all[s0:s1+1]
        az = np.rad2deg(np.arctan2(gx, gy)) % 360; rng = np.hypot(gx, gy)/ce
        out[s0:s1+1] = sample_native(az, rng, sweep_sampler(rSrc, k, field))
    return out

truth = np.ma.filled(radars["t2"].fields["reflectivity"]["data"].astype(float), np.nan)
t1_on2 = resample_to_geometry(radars["t1"], radars["t2"])
t3_on2 = resample_to_geometry(radars["t3"], radars["t2"])
lin2   = np.nanmean([t1_on2, t3_on2], axis=0)

def score(pred, name):
    m = np.isfinite(pred) & np.isfinite(truth) & (truth > 5)
    e = pred[m]-truth[m]
    cc = np.corrcoef(pred[m], truth[m])[0,1]
    print(f"{name:28s} RMSE {np.sqrt((e**2).mean()):.2f}  MAE {np.abs(e).mean():.2f}  "
          f"bias {e.mean():+.2f}  CC {cc:.3f}  (n={m.sum():,})")

score(t1_on2,    "persistence t1->2")
score(t3_on2,    "persistence t3->2")
score(lin2,      "linear avg (no advect)")
score(R2_native, "advection morph (native)")

## 6. Filling the gaps — a smooth animation

To turn the interpolation into a smooth animation we synthesize intermediate frames between
each pair of real volumes. Two practical choices:

- **Pairwise flow.** Compute optical flow on *consecutive* volumes (t₁→t₂ and t₂→t₃
  separately, ~6 km displacement each) rather than across the full t₁→t₃ span. Shorter
  intervals give cleaner flow and more accurate tweening.
- **Honest labelling.** Every synthetic frame is tagged *interpolated* (red) and every real
  scan *OBSERVED volume* (black), so the animation never disguises reconstruction as
  observation.

Here we insert **5 evenly-spaced tween frames per gap** (α = 1/6 … 5/6), rendered on the
native 0.75° PPI. Played back, the ~7-minute gaps between real scans become a continuous
progression of the squall line.


In [ ]:
import copy
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def build_frames(n_tween=5):
    """Ordered list of (reflectivity_2d, time, is_observed) for the full sequence."""
    echo12 = (Z1c > 5) | (Z2c > 5); echo23 = (Z2c > 5) | (Z3c > 5)
    fU12, fV12 = build_flow_interpolators(*level_displacements(Z1c, Z2c), echo12)
    fU23, fV23 = build_flow_interpolators(*level_displacements(Z2c, Z3c), echo23)
    alphas = [(i+1)/(n_tween+1) for i in range(n_tween)]
    getR = lambda k: np.ma.filled(radars[k].fields["reflectivity"]["data"], np.nan)
    frames = [(getR("t1"), midtimes["t1"], True)]
    for a in alphas:
        frames.append((interp_native_volume(radars["t1"], radars["t2"], fU12, fV12, a),
                       midtimes["t1"]+datetime.timedelta(seconds=a*dt12), False))
    frames.append((getR("t2"), midtimes["t2"], True))
    for a in alphas:
        frames.append((interp_native_volume(radars["t2"], radars["t3"], fU23, fV23, a),
                       midtimes["t2"]+datetime.timedelta(seconds=a*dt23), False))
    frames.append((getR("t3"), midtimes["t3"], True))
    return frames

frames = build_frames(n_tween=5)
print(f"{len(frames)} frames  (3 observed + {len(frames)-3} interpolated)")

In [ ]:
SWEEP = 0
base = radars["t1"]; s0, s1 = base.get_start_end(SWEEP)
gx = base.gate_x["data"][s0:s1+1]/1000; gy = base.gate_y["data"][s0:s1+1]/1000
sweep_field = lambda A: np.ma.masked_less(np.ma.masked_invalid(A[s0:s1+1]), -5.0)

fig, ax = plt.subplots(figsize=(8.6, 8.0), constrained_layout=True)
pm = ax.pcolormesh(gx, gy, sweep_field(frames[0][0]), vmin=-8, vmax=64,
                   cmap="ChaseSpectral", shading="auto")
ax.set_aspect("equal"); ax.set_xlim(-120, 120); ax.set_ylim(-120, 120)
ax.set_xlabel("x (km E of radar)"); ax.set_ylabel("y (km N of radar)")
th = np.linspace(0, 2*np.pi, 200)
for rr in (50, 100): ax.plot(rr*np.cos(th), rr*np.sin(th), color="0.6", lw=0.5)
fig.colorbar(pm, ax=ax, shrink=0.85, label="reflectivity (dBZ)")
title = ax.set_title("")
tag = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top", fontsize=11,
              fontweight="bold", bbox=dict(boxstyle="round", fc="white", ec="0.5", alpha=0.85))

def update(i):
    A, t, obs = frames[i]
    pm.set_array(sweep_field(A).ravel())
    title.set_text(f"C-SAPR 0.75° PPI — native advection interpolation\n{t:%Y-%m-%d %H:%M:%S} UTC")
    tag.set_text("OBSERVED volume" if obs else "interpolated")
    tag.set_color("black" if obs else "#b30000")
    return pm, title, tag

anim = FuncAnimation(fig, update, frames=len(frames), blit=False)
import os
os.makedirs("figures", exist_ok=True)
anim.save("figures/ppi_interp_animation.gif", writer=PillowWriter(fps=4))
plt.close(fig)
print("saved figures/ppi_interp_animation.gif")

![PPI advection-interpolation animation](figures/ppi_interp_animation.gif)

*Native 0.75° PPI. Black-tagged frames are real C-SAPR volumes; red-tagged frames are
advection-morphed reconstructions filling the gaps.*

## 7. Building a deliverable CfRadial volume

The reconstructed field can be written back into a Py-ART `Radar` object (a deep copy of the
real t₂ geometry, with reflectivity replaced by the reconstruction) and saved as CfRadial —
an ordinary radar volume that any Py-ART-aware tool can read.


In [ ]:
recon_radar = copy.deepcopy(radars["t2"])
recon_radar.fields["reflectivity"]["data"] = np.ma.masked_invalid(R2_native)
# keep the real field alongside for comparison
recon_radar.add_field("reflectivity_truth",
                      copy.deepcopy(radars["t2"].fields["reflectivity"]), replace_existing=True)
pyart.io.write_cfradial("csapr_t2_reconstructed.nc", recon_radar)
print("wrote csapr_t2_reconstructed.nc")

## Summary

| step | choice | why |
|---|---|---|
| motion estimation | dense TV-L1 optical flow, per height level | a single rigid vector fails for an MCS; motion varies in space and with height |
| interpolation | semi-Lagrangian morph, forward/backward warp + time blend | places moving echo correctly at the target time |
| **sign** | forward-warp samples source at **+α·D** | verified end-to-end (warping t₁ by +D reproduces t₃) — the easiest thing to get wrong |
| geometry | **native (az, range, elevation), gate-by-gate** | output is a true radar volume, not a grid |
| animation | pairwise flow, 5 tweens/gap, observed vs interpolated labelled | smooth playback that stays honest about what's reconstructed |

The morph beats naïve averaging and persistence on every error metric. Its advantage is
largest at far range, where gates are large and the storm moves several gate-widths between
scans. The main residual error is storm growth and decay — an intensity change the kinematic
morph cannot represent.

**Extensions.** Apply the same morph to the other moments (ZDR, KDP, ρhv; velocity needs
care as a vector component); test across a wider range of storm speeds and scan cadences;
and blend in an intensity-evolution term to capture growth/decay at cell edges.

---
*C-SAPR data: U.S. Department of Energy ARM user facility, MC3E campaign. Method and notebook
prepared for the notebook collection — see repository README.*
